# 🏥 Agent-D v0 — Domain-Adaptive SFT (Structural Diversity Project)

**Goal:** Fine-tune Qwen2.5-3B-Instruct in two phases:
1. **Phase 1** — Domain-adaptive continued pre-training on PubMed abstracts (1 epoch, LoRA r=8)
2. **Phase 2** — Instruction SFT on MedQA + PubMedQA (2 epochs, LoRA r=16)

**Output:** `Agent-D-v0` checkpoint — pre-co-evolution starting point for the domain specialist.

---

### ⚙️ Kaggle Setup (READ BEFORE RUNNING)

| Setting | Value |
|---------|-------|
| **GPU** | **T4 x2** — select in Kaggle settings. Gives ~30 GB VRAM total. Do NOT use P100 (single 16 GB, will OOM during Phase 2). |
| **Internet** | ON — needed for HuggingFace dataset/model downloads |
| **Persistence** | Keep session alive. Total runtime: ~3.5–4.5 hrs |
| **Your Kaggle datasets** | Add your Qwen2.5-3B-Instruct dataset before running. If you already have it mounted from the Agent-R run, path is the same. |

> **Why T4 x2, not P100?**  
> Phase 2 runs both MedQA and PubMedQA through the model simultaneously. With 4-bit QLoRA, you need ~14 GB per device. T4 x2 gives you two 16 GB GPUs and lets `device_map='auto'` distribute safely. P100 is one 16 GB card — tight and fragile for batch size > 1.

---

### 📦 Kaggle Datasets to Add
- Your `Qwen2.5-3B-Instruct` model files dataset  
- *(Optional)* Your PubMed abstracts dataset if pre-downloaded — otherwise we stream from HuggingFace

In [ ]:
# ── Cell 1 — Install Dependencies ─────────────────────────────────────────
# Same versions as Agent-R to avoid environment drift between agents
!pip install -q \
    trl==1.2.0 \
    transformers==4.47.0 \
    accelerate==1.7.0 \
    peft==0.15.2 \
    bitsandbytes==0.46.0 \
    datasets==4.7.0 \
    einops \
    sentence-transformers

print('✅ Dependencies installed')

In [ ]:
# ── Cell 2 — Imports & GPU Check ───────────────────────────────────────────
import os, re, json, time, warnings
warnings.filterwarnings('ignore')

import torch
from transformers import (
    AutoTokenizer, AutoModelForCausalLM,
    BitsAndBytesConfig, TrainerCallback,
    DataCollatorForLanguageModeling,
)
from peft import LoraConfig, get_peft_model, PeftModel
from datasets import load_dataset, concatenate_datasets, Dataset
from trl import SFTTrainer, SFTConfig

print(f'PyTorch      : {torch.__version__}')
print(f'CUDA available: {torch.cuda.is_available()}')
print(f'GPU count    : {torch.cuda.device_count()}')
for i in range(torch.cuda.device_count()):
    p = torch.cuda.get_device_properties(i)
    print(f'  GPU {i}: {p.name} — {p.total_memory/1e9:.1f} GB')

if torch.cuda.device_count() < 2:
    print('\n⚠️  WARNING: Only 1 GPU detected. Switch Kaggle accelerator to T4 x2.')
    print('   Settings → Accelerator → GPU T4 x2')
else:
    print('\n✅ T4 x2 confirmed — good to go')

In [ ]:
# ── Cell 3 — HuggingFace Login (optional but recommended) ─────────────────
# Required if: 
#   - You want to push the final adapter to HF Hub
#   - Any dataset you use is gated (MedQA from bigbio requires agreement)
# Skip this cell if you're loading everything from Kaggle datasets.

USE_HF_LOGIN = True   # ← set False to skip

if USE_HF_LOGIN:
    from huggingface_hub import login
    # Add your HF token to Kaggle Secrets:
    #   Add-ons → Secrets → New Secret → name: HF_TOKEN
    from kaggle_secrets import UserSecretsClient
    try:
        secrets = UserSecretsClient()
        hf_token = secrets.get_secret('HF_TOKEN')
        login(token=hf_token, add_to_git_credential=False)
        print('✅ HuggingFace login successful')
    except Exception as e:
        print(f'⚠️  HF login failed: {e}')
        print('   Continuing without login — will use public datasets only')
else:
    print('Skipping HF login')

In [ ]:
# ── Cell 4 — Paths Configuration ──────────────────────────────────────────
import glob

# ── Base model ─────────────────────────────────────────────────────────────
# This should be the SAME path you used for Agent-R on Kaggle
BASE_MODEL_PATH = '/kaggle/input/qwen2-5-3b-instruct'   # ← adjust to your dataset mount

# ── Output directories ──────────────────────────────────────────────────────
OUTPUT_DIR_P1  = '/kaggle/working/agent_d_phase1_pretrain'   # after Phase 1
OUTPUT_DIR_P2  = '/kaggle/working/agent_d_phase2_sft'        # after Phase 2
FINAL_CKPT_DIR = '/kaggle/working/agent_d_v0'                # final Agent-D-v0

for d in [OUTPUT_DIR_P1, OUTPUT_DIR_P2, FINAL_CKPT_DIR]:
    os.makedirs(d, exist_ok=True)

# ── Sanity check ───────────────────────────────────────────────────────────
if not os.path.isdir(BASE_MODEL_PATH):
    # Try to auto-find it
    candidates = glob.glob('/kaggle/input/*qwen*', recursive=False)
    if candidates:
        BASE_MODEL_PATH = candidates[0]
        print(f'Auto-found model at: {BASE_MODEL_PATH}')
    else:
        raise FileNotFoundError(
            f'Base model not found. Add your Qwen2.5-3B-Instruct Kaggle dataset.'
        )

print(f'✅ Base model : {BASE_MODEL_PATH}')
print(f'✅ Phase 1 out: {OUTPUT_DIR_P1}')
print(f'✅ Phase 2 out: {OUTPUT_DIR_P2}')
print(f'✅ Final ckpt : {FINAL_CKPT_DIR}')

In [ ]:
# ── Cell 5 — Load Tokenizer + Base Model (4-bit QLoRA) ────────────────────
# We load ONCE and reuse across both phases. The LoRA adapter is swapped between phases.

print('Loading tokenizer...')
tokenizer = AutoTokenizer.from_pretrained(
    BASE_MODEL_PATH,
    trust_remote_code=True,
)
tokenizer.pad_token     = tokenizer.eos_token
tokenizer.padding_side  = 'right'   # SFT uses right-padding (unlike GRPO)

print('Loading base model in 4-bit...')
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type='nf4',
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_use_double_quant=True,
)

base_model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL_PATH,
    quantization_config=bnb_config,
    device_map='auto',
    trust_remote_code=True,
    attn_implementation='sdpa',
)
base_model.config.use_cache = False   # required for gradient checkpointing

print(f'✅ Model loaded | dtype: {next(base_model.parameters()).dtype}')
print(f'   Params: {sum(p.numel() for p in base_model.parameters())/1e9:.2f}B total')

---
## Phase 1 — Domain-Adaptive Continued Pre-Training on PubMed

**What this does:** Exposes Qwen to raw biomedical text (PubMed abstracts) with standard causal LM loss — no instruction formatting. This shifts the model's prior toward medical language before we apply instruction tuning.

**Why LoRA r=8 here (not r=16):** Pre-training phase just needs to shift the distribution slightly. A smaller rank is faster, uses less memory, and avoids over-specialising before the instruction SFT phase.

**Dataset:** We use `ccdv/pubmed-summarization` — it has 133k PubMed article abstracts, clean and HF-ready. We only need the `article` (abstract) column.

In [ ]:
# ── Cell 6 — Phase 1 Dataset: PubMed Abstracts ────────────────────────────
print('Loading PubMed abstracts...')

# Option A: stream from HuggingFace (uses internet, ~2 min)
pubmed_raw = load_dataset('ccdv/pubmed-summarization', 'document', split='train')

# Option B: if you have a Kaggle PubMed dataset mounted, change to:
# pubmed_raw = load_dataset('json', data_files='/kaggle/input/your-pubmed-dataset/*.json')['train']

print(f'Raw PubMed records: {len(pubmed_raw)}')
print(f'Columns: {pubmed_raw.column_names}')
print(f'Sample abstract (first 200 chars):\n{pubmed_raw[0]["article"][:200]}')

# ── Subsample for compute budget ──────────────────────────────────────────
# Full dataset = 133k docs ≈ 8+ hours. We cap at 30k which covers the
# key medical vocabulary shifts in ~1 hr on T4 x2.
# If you have more time / want richer pretraining, increase to 60k.
MAX_PRETRAIN_DOCS = 30_000
pubmed_subset = pubmed_raw.shuffle(seed=42).select(range(MAX_PRETRAIN_DOCS))

# ── Format: raw text only, no instruction wrapper ─────────────────────────
MAX_SEQ_LEN_P1 = 512   # abstracts are short; 512 covers 95%+ without truncation

def tokenize_pretrain(batch):
    """Tokenise raw abstract text for causal LM pretraining."""
    return tokenizer(
        batch['article'],
        truncation=True,
        max_length=MAX_SEQ_LEN_P1,
        padding=False,   # DataCollator handles padding
    )

pretrain_ds = pubmed_subset.map(
    tokenize_pretrain,
    batched=True,
    remove_columns=pubmed_subset.column_names,
    desc='Tokenising PubMed abstracts',
)

print(f'\n✅ Phase 1 dataset ready: {len(pretrain_ds)} examples')

In [ ]:
# ── Cell 7 — Phase 1 LoRA Config (r=8, lightweight) ───────────────────────
from peft import prepare_model_for_kbit_training

# Prepare model for QLoRA training
model_p1 = prepare_model_for_kbit_training(
    base_model,
    use_gradient_checkpointing=True,
)

lora_config_p1 = LoraConfig(
    r=8,                 # smaller rank for pre-training phase
    lora_alpha=16,
    lora_dropout=0.05,
    target_modules=['q_proj', 'k_proj', 'v_proj', 'o_proj'],
    bias='none',
    task_type='CAUSAL_LM',
)

model_p1 = get_peft_model(model_p1, lora_config_p1)
model_p1.print_trainable_parameters()
# Expected: ~0.5% trainable (r=8 on 3B model)

print('\n✅ Phase 1 LoRA ready')

In [ ]:
# ── Cell 8 — Phase 1 Training Config ──────────────────────────────────────
from transformers import TrainingArguments, Trainer

# Data collator for causal LM (handles padding + sets labels = input_ids)
data_collator_p1 = DataCollatorForLanguageModeling(
    tokenizer=tokenizer,
    mlm=False,   # causal LM, not masked
)

training_args_p1 = TrainingArguments(
    output_dir=OUTPUT_DIR_P1,

    # Training schedule — 1 epoch over 30k docs
    num_train_epochs=1,
    per_device_train_batch_size=4,
    gradient_accumulation_steps=4,   # effective batch = 4×4×2GPUs = 32

    # Optimiser
    learning_rate=2e-4,
    lr_scheduler_type='cosine',
    warmup_ratio=0.05,
    optim='paged_adamw_8bit',

    # Precision
    bf16=True,
    gradient_checkpointing=True,

    # Logging + saving
    logging_steps=50,
    save_steps=500,
    save_total_limit=2,
    report_to='none',

    # Misc
    dataloader_pin_memory=False,
    remove_unused_columns=False,
    group_by_length=True,   # speeds up training by batching similar-length sequences
)

trainer_p1 = Trainer(
    model=model_p1,
    args=training_args_p1,
    train_dataset=pretrain_ds,
    data_collator=data_collator_p1,
    tokenizer=tokenizer,
)

print('✅ Phase 1 trainer ready')

In [ ]:
# ── Cell 9 — Run Phase 1 Training 🚀 ──────────────────────────────────────
print('=' * 60)
print('PHASE 1: PubMed Domain-Adaptive Pre-Training')
print(f'  Docs       : {len(pretrain_ds):,}')
print(f'  Batch size : {training_args_p1.per_device_train_batch_size} × {training_args_p1.gradient_accumulation_steps} steps × 2 GPUs = 32 effective')
print(f'  Est. time  : ~60–90 min on T4 x2')
print('=' * 60)

t0 = time.time()
trainer_p1.train()
elapsed = time.time() - t0
print(f'\n✅ Phase 1 done in {elapsed/60:.1f} min')

# Save Phase 1 adapter
model_p1.save_pretrained(OUTPUT_DIR_P1)
tokenizer.save_pretrained(OUTPUT_DIR_P1)
print(f'✅ Phase 1 adapter saved → {OUTPUT_DIR_P1}')

---
## Phase 2 — Instruction SFT on MedQA + PubMedQA

**What this does:** Fine-tune the Phase-1-adapted model on medical QA pairs with instruction format. This is what makes Agent-D an actual *question-answering agent*, not just a model that knows medical vocabulary.

**Datasets:**
- **MedQA (USMLE)** — 10k multiple-choice questions from US Medical Licensing Exam. Requires step-by-step clinical reasoning.
- **PubMedQA** — 1k labeled QA pairs from biomedical literature. Yes/No/Maybe answers with supporting evidence.

**Why both?** MedQA trains the model to do multi-choice reasoning. PubMedQA trains it to reason from literature passages. Together they cover the two distinct failure modes we want Agent-D to exhibit vs Agent-R.

In [ ]:
# ── Cell 10 — Phase 2 Dataset Preparation ─────────────────────────────────

SYSTEM_D = (
    'You are a specialist in biomedical literature and clinical medicine. '
    'Reason carefully from medical evidence before answering. '
    'Be precise about what the evidence supports.'
)

# ── MedQA (USMLE 4-option MCQ) ────────────────────────────────────────────
print('Loading MedQA...')
medqa_raw = load_dataset('GBaker/MedQA-USMLE-4-options', split='train')
print(f'MedQA train: {len(medqa_raw)} | columns: {medqa_raw.column_names}')
print(f'Sample: {medqa_raw[0]}')

In [ ]:
# ── Cell 11 — Format MedQA ─────────────────────────────────────────────────

# MedQA columns: 'question', 'options' (dict: A/B/C/D → text), 'answer_idx', 'answer'
# We format as: question + options → answer with brief explanation template

def format_medqa(example):
    """Convert MedQA row to chat-formatted instruction pair."""
    # Build options string
    options_text = '\n'.join(
        f'{k}. {v}' for k, v in example['options'].items()
    )
    user_msg = (
        f"{example['question']}\n\n"
        f"Options:\n{options_text}\n\n"
        f"Choose the best answer and explain your reasoning."
    )
    # Correct answer letter + text
    answer_letter = example['answer_idx']       # 'A', 'B', 'C', or 'D'
    answer_text   = example['options'][answer_letter]
    assistant_msg = (
        f"The correct answer is **{answer_letter}. {answer_text}**.\n\n"
        f"Rationale: This follows from standard clinical principles. "
        f"Option {answer_letter} is correct because it best addresses the "
        f"clinical scenario presented."
    )
    messages = [
        {'role': 'system',    'content': SYSTEM_D},
        {'role': 'user',      'content': user_msg},
        {'role': 'assistant', 'content': assistant_msg},
    ]
    return {
        'text': tokenizer.apply_chat_template(
            messages, tokenize=False, add_generation_prompt=False
        )
    }

medqa_ds = medqa_raw.map(
    format_medqa,
    remove_columns=medqa_raw.column_names,
    desc='Formatting MedQA',
)

print(f'\n✅ MedQA formatted: {len(medqa_ds)} examples')
print(f'Sample (first 500 chars):\n{medqa_ds[0]["text"][:500]}')

In [ ]:
# ── Cell 12 — Load and Format PubMedQA ────────────────────────────────────
print('Loading PubMedQA...')
pubmedqa_raw = load_dataset('qiaojin/PubMedQA', 'pqa_labeled', split='train')
print(f'PubMedQA: {len(pubmedqa_raw)} | columns: {pubmedqa_raw.column_names}')
print(f'Sample final_decision values: {set(pubmedqa_raw["final_decision"])}')

def format_pubmedqa(example):
    """Convert PubMedQA row to chat-formatted instruction pair."""
    # PubMedQA columns: 'pubid', 'question', 'context' (dict with 'contexts' list),
    #                   'long_answer', 'final_decision' (yes/no/maybe)
    
    # Concatenate context passages (cap at 800 chars to avoid exceeding seq len)
    context_passages = example['context']['contexts']
    context_text     = ' '.join(context_passages)[:800]
    
    user_msg = (
        f"Based on the following research context, answer the question.\n\n"
        f"Context:\n{context_text}\n\n"
        f"Question: {example['question']}\n"
        f"Answer with yes, no, or maybe, then explain."
    )
    decision  = example['final_decision'].strip().lower()  # 'yes', 'no', 'maybe'
    long_ans  = example['long_answer'].strip()[:600]       # cap long answer
    
    assistant_msg = (
        f"**{decision.capitalize()}.**\n\n{long_ans}"
    )
    messages = [
        {'role': 'system',    'content': SYSTEM_D},
        {'role': 'user',      'content': user_msg},
        {'role': 'assistant', 'content': assistant_msg},
    ]
    return {
        'text': tokenizer.apply_chat_template(
            messages, tokenize=False, add_generation_prompt=False
        )
    }

pubmedqa_ds = pubmedqa_raw.map(
    format_pubmedqa,
    remove_columns=pubmedqa_raw.column_names,
    desc='Formatting PubMedQA',
)

print(f'\n✅ PubMedQA formatted: {len(pubmedqa_ds)} examples')

In [ ]:
# ── Cell 13 — Merge Datasets + Train/Val Split ─────────────────────────────
sft_full = concatenate_datasets([medqa_ds, pubmedqa_ds]).shuffle(seed=42)

# 95/5 train/val split (val is for early stopping monitoring only)
split = sft_full.train_test_split(test_size=0.05, seed=42)
sft_train = split['train']
sft_val   = split['test']

print(f'SFT train : {len(sft_train)} examples')
print(f'SFT val   : {len(sft_val)} examples')
print(f'Total     : {len(sft_full)} examples')

# Quick token length check — make sure sequences aren't pathologically long
sample_lengths = [
    len(tokenizer(sft_train[i]['text'], truncation=False)['input_ids'])
    for i in range(min(200, len(sft_train)))
]
import statistics
print(f'\nToken length stats (sample of 200):')
print(f'  Mean   : {statistics.mean(sample_lengths):.0f}')
print(f'  Median : {statistics.median(sample_lengths):.0f}')
print(f'  Max    : {max(sample_lengths)}')
print(f'  >1024  : {sum(l > 1024 for l in sample_lengths)} / 200')

In [ ]:
# ── Cell 14 — Load Phase 1 Adapter + Add Phase 2 LoRA ─────────────────────
# We start Phase 2 from the Phase 1 checkpoint (domain-adapted weights)

print('Loading Phase 1 adapted model...')
model_p2 = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL_PATH,
    quantization_config=bnb_config,
    device_map='auto',
    trust_remote_code=True,
    attn_implementation='sdpa',
)
model_p2.config.use_cache = False

# Merge Phase 1 LoRA into base weights cleanly
# This avoids the double-adapter stacking problem
model_p2 = PeftModel.from_pretrained(model_p2, OUTPUT_DIR_P1)
model_p2 = model_p2.merge_and_unload()   # bake P1 adapter into weights
print('Phase 1 adapter merged into base weights ✅')

# Re-prepare for QLoRA training
from peft import prepare_model_for_kbit_training
model_p2 = prepare_model_for_kbit_training(
    model_p2,
    use_gradient_checkpointing=True,
)

# Add fresh Phase 2 LoRA (r=16 — full rank for instruction tuning)
lora_config_p2 = LoraConfig(
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    target_modules=['q_proj', 'k_proj', 'v_proj', 'o_proj', 'gate_proj', 'up_proj', 'down_proj'],
    bias='none',
    task_type='CAUSAL_LM',
)
model_p2 = get_peft_model(model_p2, lora_config_p2)
model_p2.print_trainable_parameters()

print('\n✅ Phase 2 model ready (domain-adapted + fresh LoRA r=16)')

In [ ]:
# ── Cell 15 — Phase 2 SFT Training Config ─────────────────────────────────

class TrainingProgressCallback(TrainerCallback):
    """Clean per-step progress logging."""
    def __init__(self):
        self.start_time = None

    def on_train_begin(self, args, state, control, **kwargs):
        self.start_time = time.time()

    def on_log(self, args, state, control, logs=None, **kwargs):
        if logs is None or self.start_time is None:
            return
        step    = state.global_step
        total   = state.max_steps
        pct     = step / total * 100 if total else 0
        elapsed = time.time() - self.start_time
        eta     = (elapsed / step * (total - step)) if step > 0 else 0
        loss    = logs.get('loss', 'N/A')
        lr      = logs.get('learning_rate', 'N/A')
        loss_s  = f'{loss:.4f}' if isinstance(loss, float) else loss
        lr_s    = f'{lr:.2e}'   if isinstance(lr, float)   else lr
        filled  = int(pct / 2)
        bar     = '█' * filled + '░' * (50 - filled)
        print(
            f'\r[{bar}] {pct:5.1f}%  step {step}/{total}'
            f'  loss={loss_s}  lr={lr_s}  ETA={eta/60:.1f}m',
            end='', flush=True
        )
        if step == total:
            print()


MAX_SEQ_LEN_P2 = 1024  # covers 99%+ of MedQA + PubMedQA sequences

sft_config = SFTConfig(
    output_dir=OUTPUT_DIR_P2,

    # Training schedule
    num_train_epochs=2,
    per_device_train_batch_size=2,
    gradient_accumulation_steps=4,    # effective batch = 2×4×2 GPUs = 16

    # Optimiser
    learning_rate=1e-4,               # as specified in the plan
    lr_scheduler_type='cosine',
    warmup_ratio=0.05,
    optim='paged_adamw_8bit',
    weight_decay=0.01,

    # Precision + memory
    bf16=True,
    gradient_checkpointing=True,
    max_seq_length=MAX_SEQ_LEN_P2,

    # Logging + saving
    logging_steps=25,
    eval_strategy='steps',
    eval_steps=100,
    save_steps=200,
    save_total_limit=3,
    load_best_model_at_end=True,
    metric_for_best_model='eval_loss',
    greater_is_better=False,
    report_to='none',

    # Dataset
    dataset_text_field='text',
    remove_unused_columns=False,
    dataloader_pin_memory=False,
    packing=False,   # keep packing off — our sequences vary a lot in length
)

trainer_p2 = SFTTrainer(
    model=model_p2,
    args=sft_config,
    train_dataset=sft_train,
    eval_dataset=sft_val,
    processing_class=tokenizer,
    callbacks=[TrainingProgressCallback()],
)

print('✅ Phase 2 SFT trainer ready')
print(f'   Total train steps: {trainer_p2.args.max_steps if trainer_p2.args.max_steps > 0 else "auto (2 epochs)"}')

In [ ]:
# ── Cell 16 — Run Phase 2 SFT Training 🚀 ─────────────────────────────────
print('=' * 60)
print('PHASE 2: Instruction SFT on MedQA + PubMedQA')
print(f'  Train examples : {len(sft_train)}')
print(f'  Val examples   : {len(sft_val)}')
print(f'  Epochs         : 2')
print(f'  Learning rate  : 1e-4')
print(f'  Est. time      : ~2.0–2.5 hrs on T4 x2')
print('=' * 60)

t0 = time.time()
trainer_p2.train()
elapsed = time.time() - t0
print(f'\n✅ Phase 2 done in {elapsed/60:.1f} min ({elapsed/3600:.2f} hrs)')

# Final eval loss
results = trainer_p2.evaluate()
print(f'\nFinal eval loss: {results["eval_loss"]:.4f}')
print('Target: eval_loss < 1.5 for reasonable medical QA quality')

In [ ]:
# ── Cell 17 — Save Agent-D-v0 Checkpoint ──────────────────────────────────
trainer_p2.save_model(FINAL_CKPT_DIR)
tokenizer.save_pretrained(FINAL_CKPT_DIR)

meta = {
    'checkpoint_name'  : 'Agent-D-v0',
    'base_model'       : BASE_MODEL_PATH,
    'phase1'           : {
        'dataset'      : 'ccdv/pubmed-summarization (30k abstracts)',
        'epochs'       : 1,
        'lora_r'       : 8,
        'lora_alpha'   : 16,
        'learning_rate': 2e-4,
    },
    'phase2'           : {
        'datasets'     : ['GBaker/MedQA-USMLE-4-options', 'qiaojin/PubMedQA/pqa_labeled'],
        'train_size'   : len(sft_train),
        'epochs'       : 2,
        'lora_r'       : 16,
        'lora_alpha'   : 32,
        'learning_rate': 1e-4,
        'eval_loss'    : results.get('eval_loss', 'N/A'),
    },
    'agent_role'       : 'Domain specialist — biomedical literature + clinical QA',
    'failure_mode'     : 'Strong medical pattern-matching; fails on multi-step cross-domain reasoning',
    'project'          : 'Disagreement, Diversity, and Difficulty in Heterogeneous Multi-Agent Language Models',
}

with open(os.path.join(FINAL_CKPT_DIR, 'agent_d_meta.json'), 'w') as f:
    json.dump(meta, f, indent=2)

print('\n✅ Agent-D-v0 saved!')
print(f'   Path: {FINAL_CKPT_DIR}')
print(f'\n{json.dumps(meta, indent=2)}')

In [ ]:
# ── Cell 18 — Push to HuggingFace Hub (optional) ──────────────────────────
PUSH_TO_HUB = False   # ← set True if you want it on HF Hub
HF_REPO_NAME = 'your-username/agent-d-v0'   # ← change to your HF username

if PUSH_TO_HUB and USE_HF_LOGIN:
    trainer_p2.model.push_to_hub(HF_REPO_NAME)
    tokenizer.push_to_hub(HF_REPO_NAME)
    print(f'✅ Pushed to HF Hub: {HF_REPO_NAME}')
else:
    print('Skipping HF Hub push (PUSH_TO_HUB=False)')
    print('→ Download Agent-D-v0 from Kaggle Output before session ends!')

---
## Validation — Does Agent-D Actually Behave Like a Domain Expert?

We run a small qualitative + quantitative sanity check before declaring Agent-D-v0 ready for co-evolution.

In [ ]:
# ── Cell 19 — Qualitative Sanity Check ────────────────────────────────────
model_p2.eval()

test_cases = [
    {
        'question': (
            'A 45-year-old man presents with progressive dyspnea and a dry cough. '
            'CXR shows bilateral interstitial infiltrates. He works in a sandblasting facility. '
            'What is the most likely diagnosis?\n\n'
            'A. Asbestosis\nB. Silicosis\nC. Coal worker pneumoconiosis\nD. Hypersensitivity pneumonitis\n\n'
            'Choose the best answer and explain your reasoning.'
        ),
        'expected_contains': 'B',  # Silicosis
    },
    {
        'question': (
            'Based on this context: "Metformin activates AMPK and inhibits hepatic gluconeogenesis. '
            'It does not cause hypoglycaemia when used as monotherapy."\n\n'
            'Question: Does metformin cause hypoglycaemia as monotherapy? Answer yes, no, or maybe.'
        ),
        'expected_contains': 'no',
    },
]

print('Agent-D-v0 Qualitative Evaluation')
print('=' * 60)

passed = 0
for i, tc in enumerate(test_cases):
    messages = [
        {'role': 'system', 'content': SYSTEM_D},
        {'role': 'user',   'content': tc['question']},
    ]
    prompt = tokenizer.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True
    )
    inputs = tokenizer(prompt, return_tensors='pt').to(model_p2.device)
    
    with torch.no_grad():
        out = model_p2.generate(
            **inputs,
            max_new_tokens=200,
            do_sample=False,
            temperature=1.0,
            pad_token_id=tokenizer.eos_token_id,
        )
    response = tokenizer.decode(
        out[0][inputs['input_ids'].shape[1]:],
        skip_special_tokens=True
    )
    
    ok = tc['expected_contains'].lower() in response.lower()
    passed += int(ok)
    status = '✅ PASS' if ok else '❌ FAIL'
    
    print(f'\nTest {i+1}: {status}')
    print(f'Q: {tc["question"][:100]}...')
    print(f'A: {response[:300]}')
    print(f'Expected to contain: "{tc["expected_contains"]}"')
    print('-' * 60)

print(f'\nQual check: {passed}/{len(test_cases)} passed')

In [ ]:
# ── Cell 20 — Held-Out Accuracy on PubMedQA Val Set ───────────────────────
# Quick quantitative check: does Agent-D get yes/no/maybe right?

pubmedqa_test = load_dataset('qiaojin/PubMedQA', 'pqa_labeled', split='test')
pubmedqa_test = pubmedqa_test.select(range(min(100, len(pubmedqa_test))))

correct = 0
total   = len(pubmedqa_test)

model_p2.eval()
for idx, row in enumerate(pubmedqa_test):
    context_text = ' '.join(row['context']['contexts'])[:600]
    user_msg     = (
        f"Context:\n{context_text}\n\n"
        f"Question: {row['question']}\n"
        f"Answer yes, no, or maybe."
    )
    messages = [
        {'role': 'system', 'content': SYSTEM_D},
        {'role': 'user',   'content': user_msg},
    ]
    prompt = tokenizer.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True
    )
    inputs = tokenizer(prompt, return_tensors='pt').to(model_p2.device)
    
    with torch.no_grad():
        out = model_p2.generate(
            **inputs,
            max_new_tokens=50,
            do_sample=False,
            pad_token_id=tokenizer.eos_token_id,
        )
    response = tokenizer.decode(
        out[0][inputs['input_ids'].shape[1]:],
        skip_special_tokens=True
    ).lower()
    
    gold = row['final_decision'].strip().lower()
    pred_match = any(word in response[:30] for word in ['yes', 'no', 'maybe'])
    pred = next((w for w in ['yes', 'no', 'maybe'] if w in response[:30]), 'none')
    correct += int(pred == gold)
    
    if idx % 20 == 0:
        print(f'Progress: {idx}/{total} | Running acc: {correct/(idx+1):.2%}')

final_acc = correct / total
print(f'\n✅ PubMedQA held-out accuracy: {final_acc:.2%} ({correct}/{total})')
print('Baseline (majority class "yes"): ~55%')
print('Target: > 60% to confirm domain learning')

if final_acc >= 0.60:
    print('\n🟢 Agent-D-v0 VALIDATED — ready for co-evolution loop')
else:
    print('\n🟡 Below target — but check if model is converging; 2 epochs may not be enough.')
    print('   Option: add 1 more epoch (re-run Phase 2 from last checkpoint with resume_from_checkpoint)')

In [ ]:
# ── Cell 21 — Summary ─────────────────────────────────────────────────────
print('=' * 60)
print('AGENT-D-v0 TRAINING COMPLETE')
print('=' * 60)
print(f'Checkpoint   : {FINAL_CKPT_DIR}')
print(f'Eval loss    : {results.get("eval_loss", "N/A"):.4f}' if isinstance(results.get('eval_loss'), float) else 'Eval loss: see above')
print(f'PubMedQA acc : {final_acc:.2%}')
print()
print('Next steps:')
print('  1. Download Agent-D-v0 from Kaggle Output (or push to HF Hub)')
print('  2. Start Agent-C notebook (ECE+CE loss + temperature scaling)')
print('  3. Come back to Agent-R GRPO after Agent-C is done')
print('  4. Once all 3 agents are ready → run the static disagreement experiments (Table 1)')